In [40]:
# General
import pickle

# Specific
import numpy 
import tensorflow
import spektral
from spektral import transforms
from spektral.data import BatchLoader
import networkx
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from rdkit.Chem import rdForceFieldHelpers as rdFF
from matplotlib import pyplot as plt
import pandas as pd
import torch

import sys
  
# append the path of the parent directory
sys.path.append("..")

from model.model_GCN_SigmaProfile import graphDataset, GCN_Model_SP

In [41]:
#Load solute and solvent lists
solute_list = pd.read_csv('..\data\solubility_water\solute_list.csv')
solvent_list = pd.read_csv('..\data\solubility_water\solvent_list_water.csv')

#Load the GCN model weights and compile.
# Path to Model
modelPath=r'..\SigmaProfileModel\Models\MMFF_GCN.pkl'
# Load weights
with open(modelPath,'rb') as f:
      weights=pickle.load(f)
# Define architecture
architecture={'conv1_channels': weights[0].shape[1],
              'conv2_channels': weights[1].shape[1],
              'conv3_channels': weights[2].shape[1]}
# Build model
GCN=GCN_Model_SP(architecture)
# Compile model
GCN.compile()

def graph_loader(graphSet, model):
    # Create loader
    loader=BatchLoader(graphSet, batch_size=1, shuffle=False)
    # Set model shape
    #loader = torch.tensor(loader)
    model.fit(loader.load(),steps_per_epoch=loader.steps_per_epoch,epochs=1)
    # Set weights
    model.set_weights(weights)

#Build the MMFF-based molecular graphs for the SMILES strings of interest.

def get_MMFgraph_from_smiles(smiles_string, model):
#iterate for every solute
    SMILES = smiles_string
    # Obtain RDKit mol object
    molecule=Chem.MolFromSmiles(SMILES)
    # Make hydrogens explicit
    molecule=AllChem.AddHs(molecule)
    # Generate initial 3D structure of the molecule
    AllChem.EmbedMolecule(molecule)
    # Initialize MMFF props object
    prop=rdFF.MMFFGetMoleculeProperties(molecule)
    # Check if MMFF exists for entry
    if prop is None:
        raise ValueError('Cannot describe requested molecule using MMFF.')
    else:
        # Initialize container
        atomTypes=[]
        # Retrieve node-level features
        for atom in molecule.GetAtoms():
            # Get MMFF atom type
            atomType=prop.GetMMFFAtomType(atom.GetIdx())
            # Append
            atomTypes.append(atomType)

    #Build the node-level feature matrix of the graph by one-hot encoding MMFF atom types.
    # Define available unique MMFF atom types (F)
    uniqueAtomTypes=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,17,18,20,21,22,23,24,25,
                    26,27,28,29,30,31,32,33,35,37,38,39,40,42,43,44,45,59,61,63,
                    64,65,66,70,71,72,74]
    # Initialize node-level feature matrix (X) of size NxF
    X=numpy.zeros((len(atomTypes),len(uniqueAtomTypes)))
    # Iterate over atomTypes
    for n in range(len(atomTypes)):
        # Check that atom type it is available
        if atomTypes[n] not in uniqueAtomTypes:
            raise ValueError('Molecule contains atoms not trained on the GCN.')
        # Map force field atom type to index in uniqueAtomTypes
        oneHotIndex=uniqueAtomTypes.index(atomTypes[n])
        # One-hot encode atom n
        oneHot=spektral.utils.one_hot(oneHotIndex,len(uniqueAtomTypes))
        # Add entry to X
        X[n,:]=oneHot

    #Build a Spektral graph dataset object for the graph.
    # Get adjacency matrix
    adjacencyMatrix=Chem.GetAdjacencyMatrix(molecule).astype('float32')
    # Build graph
    graph=spektral.data.graph.Graph(x=X,a=adjacencyMatrix,e=None,y=numpy.ones((51,)))
    # Convert to dataset
    graphSet=graphDataset(graph)
    #print(graphSet.size())

    #Load the GCN model weights and compile.
    graph_loader(graphSet, model)

    return graphSet

C:\Users\kverg\AppData\Local\Temp\ipykernel_27024\1561681784.py:10: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  weights=pickle.load(f)


In [42]:
graphSet = get_MMFgraph_from_smiles('c1ccc(cc1)C[C@@H](C(=O)O)N', GCN)
# Apply filters to adjacency matrices
graphSet.apply(transforms.GCNFilter())
# Predict sigma profile
loader=BatchLoader(graphSet,shuffle=False)
predSP=GCN.predict(loader.load(),steps=loader.steps_per_epoch)

ValueError: Unrecognized data type: x=<spektral.data.loaders.BatchLoader object at 0x000001221D26A2E0> (of type <class 'spektral.data.loaders.BatchLoader'>)

In [ ]:
sigma=numpy.linspace(-0.025,0.025,51)
plt.plot(sigma,predSP[0,:],'--*k')
plt.xlabel(r'$\rm\sigma$ $\rm/e\cdotÅ^{2}$')
plt.ylabel(r'$\rm P(\sigma) \cdot A$ $\rm/Å^{2}$')
plt.show()